# 2 - Hybrid pipeline

The **hybrid** configuration takes the observed Doppler and the geometry from the operational Sentinel-1 ocean product, then applies the same sideband, mispointing, Stokes and wave corrections. It runs on the ocean-product grid (no deramping of our own), so comparing it with the custom pipeline isolates the effect of the deramping and Doppler estimation.

In [ ]:
import os, sys
from pathlib import Path
# resolve repo root whether launched from notebooks/ or the repo root
REPO = os.getcwd()
if not os.path.exists(os.path.join(REPO, 'scripts', 'sentinel_1')):
    REPO = os.path.abspath(os.path.join(REPO, '..'))
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np
import matplotlib.pyplot as plt
from scripts.run_pipeline import scene_paths, CUSTOM_CFG
from scripts.sentinel_1.grid_merge import merge_burst_grids
print('repo root:', REPO)

## Run the hybrid configuration

In [ ]:
from scripts.sentinel_1.pipeline import run_all_bursts
paths = scene_paths('data', scene='scene1', subswath='iw1', pol='vv')
BURSTS = run_all_bursts(**paths, use_ocn_dc=True)
print('fields:', sorted(k for k in BURSTS[0] if hasattr(BURSTS[0][k], 'shape')))

## Observed Doppler from the ocean product (`rvlDcObs`)

In [ ]:
glat, glon, g = merge_burst_grids(BURSTS, variable='f_dc', overlap='average', resolution_deg=0.01)
v = g[np.isfinite(g)]; m = np.nanpercentile(np.abs(v), 98)
fig, ax = plt.subplots(figsize=(6, 6.4), constrained_layout=True)
im = ax.pcolormesh(glon, glat, g, cmap='RdBu_r', vmin=-m, vmax=m, shading='auto')
ax.set_title('Observed Doppler centroid (ocean product)'); ax.set_xlabel('lon'); ax.set_ylabel('lat')
fig.colorbar(im, ax=ax, shrink=0.9, label='Doppler [Hz]'); plt.show()
print('mean %.3f  range [%.3f, %.3f] m/s' % (v.mean(), v.min(), v.max()))

## Hybrid surface current

In [ ]:
glat, glon, g = merge_burst_grids(BURSTS, variable='v_current_ocn', overlap='average', resolution_deg=0.01)
v = g[np.isfinite(g)]; m = np.nanpercentile(np.abs(v), 98)
fig, ax = plt.subplots(figsize=(6, 6.4), constrained_layout=True)
im = ax.pcolormesh(glon, glat, g, cmap='RdBu_r', vmin=-m, vmax=m, shading='auto')
ax.set_title('Hybrid pipeline: surface radial current'); ax.set_xlabel('lon'); ax.set_ylabel('lat')
fig.colorbar(im, ax=ax, shrink=0.9, label='radial current [m/s]'); plt.show()
print('mean %.3f  range [%.3f, %.3f] m/s' % (v.mean(), v.min(), v.max()))

> Command-line equivalent: `python scripts/run_pipeline.py --hybrid`